In [0]:
%run ./secrets-template

In [0]:
%python
import os

ssh_priv_key = dbutils.secrets.get(scope="brev", key="ssh_private_key")

with open("/tmp/ssh_private_key_4", "w") as f:
    f.write(ssh_priv_key + "\n")
os.chmod("/tmp/ssh_private_key_4", 0o600)

ssh_user = dbutils.secrets.get(scope='brev', key='ssh_user').strip().splitlines()[-1]
os.environ['SSH_USER'] = ssh_user
print(f'SSH_USER: {ssh_user}')

In [0]:
%sh
ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << EOF
export PATH="\$HOME/.local/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin"; # uv/uvx
export UV_CACHE_DIR=/ephemeral/cache/uv
# Non-interactive HF login. 'hf auth login' reads the token from stdin, so a pipe
# is unreliable -> use --token. Writes ~/.cache/huggingface/token, which any
# huggingface_hub install (incl. the converter below) then picks up automatically.
uvx --from huggingface_hub hf auth login --token "$HF_TOKEN" --add-to-git-credential
uvx --from huggingface_hub hf auth whoami
EOF

In [0]:
%sh
ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << EOF
set -e
export PATH="\$HOME/.local/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin"; # uv
export UV_CACHE_DIR=/ephemeral/cache/uv
export UV_LINK_MODE=copy

# 1. Make sure the GR00T env is ready
cd \$HOME/Isaac-GR00T
uv sync --python 3.10
.venv/bin/python -c "import gr00t; print('gr00t:', gr00t.__file__)"

# 2. Install LeRobot into GR00T's .venv at the pinned converter commit.
# NOTE: uv venvs ship no pip (uv sync even removes it), so install with
# 'uv pip install --python <venv>/bin/python', not '.venv/bin/python -m pip'.
if [ ! -d \$HOME/lerobot ]; then
  git clone https://github.com/huggingface/lerobot.git \$HOME/lerobot
fi
cd \$HOME/lerobot
git fetch --all
git checkout f25ac02
uv pip install --python \$HOME/Isaac-GR00T/.venv/bin/python -e . --no-deps

# Converter dependencies
uv pip install --python \$HOME/Isaac-GR00T/.venv/bin/python -U huggingface_hub jsonlines pyarrow numpy tqdm

# ffmpeg (video decode for the converter)
sudo apt-get update
sudo apt-get install -y ffmpeg

\$HOME/Isaac-GR00T/.venv/bin/python -c "import lerobot; print('lerobot:', lerobot.__file__)"

# 3+4. Run the converter. It downloads the v3 dataset from HF (repo id
# \$HF_USER/\$DATASET_NAME) and writes the v2.1 dataset under
# /ephemeral/$HF_USER/$DATASET_NAME. Use .venv/bin/python, NOT 'uv run python'.
cd \$HOME/Isaac-GR00T
.venv/bin/python scripts/lerobot_conversion/convert_v3_to_v2.py \\
  --repo-id "$HF_USER/$DATASET_NAME" \\
  --root /ephemeral
EOF

In [0]:
%sh
ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << EOF
set -e
# Transcode dataset videos AV1 -> H.264. The A100 has no AV1 NVDEC, so AV1 frames
# decode on CPU and starve the GPU during training (GPU util ~10%, CPU load >2x cores).
# H.264 decodes far cheaper. Re-encode is lossy but at crf 18 visually ~lossless;
# Keep GOP=2 (-g 2, like LeRobot's AV1 encoding) so nearly every frame is a
# keyframe -- training reads one random frame per camera, and a large GOP makes
# each seek decode up to ~250 frames, starving the GPU (sawtooth util).
# resolution, fps and frame count are preserved, so LeRobot timestamp/index lookups
# stay valid. Only the .mp4 pixels change -- parquet data and labels are untouched.
VID=/ephemeral/$HF_USER/$DATASET_NAME/videos
echo "transcoding mp4s under \$VID";
command -v ffmpeg >/dev/null || sudo apt-get install -y ffmpeg
n=0
find "\$VID" -name '*.mp4' | while read -r f; do
  tmp="\${f%.mp4}.h264.mp4"
  ffmpeg -y -loglevel error -i "\$f" -c:v libx264 -crf 18 -preset fast \
      -g 2 -keyint_min 2 -sc_threshold 0 -pix_fmt yuv420p -an "\$tmp" \
    && mv -f "\$tmp" "\$f"
  n=\$((n+1)); echo "[\$n] \$f"
done
echo "== verify codec of one file (expect h264) ==";
find "\$VID" -name '*.mp4' | head -1 | xargs -r ffprobe -v error -select_streams v:0 \
  -show_entries stream=codec_name,nb_frames,avg_frame_rate -of default=nk=0
echo "done"
EOF

In [0]:
%sh
ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << EOF
mkdir -p /ephemeral/$HF_USER/$DATASET_NAME/meta/
# modality.json: download the file named by the secret, then rename it to
# modality.json (the fixed name GR00T's LeRobot loader reads from meta/).
# Lets the Databricks volume hold per-dataset filenames without collisions.
curl -sL -o /ephemeral/$HF_USER/$DATASET_NAME/meta/$MODALITY_JSON \
  -H "Authorization: Bearer $DATABRICKS_TOKEN" \
  "$DATABRICKS_HOST/api/2.0/fs/files$MODALITY_FILES_PATH/$MODALITY_JSON"
mv -f /ephemeral/$HF_USER/$DATASET_NAME/meta/$MODALITY_JSON /ephemeral/$HF_USER/$DATASET_NAME/meta/modality.json
# modality.py: kept under its secret name; training passes it via --modality_config_path.
curl -sL -o /ephemeral/$HF_USER/$DATASET_NAME/meta/$MODALITY_PY \
  -H "Authorization: Bearer $DATABRICKS_TOKEN" \
  "$DATABRICKS_HOST/api/2.0/fs/files$MODALITY_FILES_PATH/$MODALITY_PY"
ls -la /ephemeral/$HF_USER/$DATASET_NAME/meta/
EOF

In [0]:
%sh
ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << EOF
export PATH="\$HOME/.local/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin"; # uv
export WANDB_API_KEY=$WANDB_API_KEY
uv run wandb login
cd \$HOME/Isaac-GR00T
EOF

In [0]:
%sh
ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << EOF
cd \$HOME/Isaac-GR00T
rm -f \$HOME/TRAINING_DONE \$HOME/TRAINING_FAILED
tmux kill-session -t finetune 2>/dev/null || true
tmux new-session -d -s finetune
tmux send-keys -t finetune 'export WANDB_API_KEY=$WANDB_API_KEY' C-m
tmux send-keys -t finetune 'export WANDB_NAME=gr00t-n1d6-so100' C-m
tmux send-keys -t finetune 'export PATH=\$HOME/.local/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin' C-m
tmux send-keys -t finetune 'cd \$HOME/Isaac-GR00T && CUDA_VISIBLE_DEVICES=0 uv run python gr00t/experiment/launch_finetune.py --base_model_path nvidia/GR00T-N1.7-3B --dataset_path /ephemeral/$HF_USER/$DATASET_NAME --modality_config_path /ephemeral/$HF_USER/$DATASET_NAME/meta/$MODALITY_PY --embodiment_tag NEW_EMBODIMENT --num_gpus 1 --output_dir /ephemeral/finetuned-models/$DATASET_NAME --save_steps $SAVE_STEPS --max_steps $MAX_STEPS --use-wandb --warmup_ratio 0.05 --weight_decay 1e-5 --learning_rate 1e-4 --global_batch_size 64 --color_jitter_params brightness 0.3 contrast 0.4 saturation 0.5 hue 0.08 --dataloader_num_workers 16 2>&1 | tee \$HOME/finetune.log && touch \$HOME/TRAINING_DONE || touch \$HOME/TRAINING_FAILED' C-m
#tmux send-keys -t finetune 'exit' C-m
EOF

In [0]:
%sh
# Only set up brev auto-delete if the delete_instance secret is "true".
# Skipping keeps brev creds OFF the VM entirely when you're just testing.
if [ "$(printf '%s' "$DELETE_INSTANCE" | tr '[:upper:]' '[:lower:]')" != "true" ]; then
  echo "delete_instance=$DELETE_INSTANCE -> skipping brev install/creds (instance will be KEPT)"
  exit 0
fi
# Install brev CLI on the VM and restore ~/.brev/credentials.json from the secret,
# so the detached poll+upload script can `brev delete` the instance after upload.
# The refresh token is non-rotating, so this stored credential keeps auto-refreshing.
B64=$(printf '%s' "$BREV_CREDENTIALS_JSON" | base64 | tr -d '\n')
ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << EOF
export PATH="\$HOME/.local/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin"
if ! command -v brev >/dev/null; then
  curl -fsSL https://raw.githubusercontent.com/brevdev/brev-cli/main/bin/install-latest.sh -o /tmp/install-brev.sh
  chmod +x /tmp/install-brev.sh
  /tmp/install-brev.sh
fi
mkdir -p \$HOME/.brev; chmod 700 \$HOME/.brev
printf '%s' "$B64" | base64 -d > \$HOME/.brev/credentials.json
chmod 600 \$HOME/.brev/credentials.json
echo "brev: \$(brev --version 2>&1 | head -1)"
echo "creds restored; instance visible to brev:"
brev ls 2>&1 | head -3
EOF

In [0]:
%sh
ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << EOF
# Copy the poll-and-upload script from the Databricks volume to the VM.
curl -sL -o \$HOME/poll_and_upload_hf.sh \
  -H "Authorization: Bearer $DATABRICKS_TOKEN" \
  "$DATABRICKS_HOST/api/2.0/fs/files/Volumes/workspace/default/hdf52lerobot_script_files_metrics/poll_and_upload_hf.sh"
chmod +x \$HOME/poll_and_upload_hf.sh
echo "== head of poll_and_upload_hf.sh =="; head -8 \$HOME/poll_and_upload_hf.sh
EOF

In [0]:
%sh
# delete_instance secret gates auto-delete: pass the instance name only if true.
INST=""
if [ "$(printf '%s' "$DELETE_INSTANCE" | tr '[:upper:]' '[:lower:]')" = "true" ]; then
  INST="$BREV_INSTANCE_NAME"; echo "delete_instance=true -> instance WILL be deleted after upload"
else
  echo "delete_instance=$DELETE_INSTANCE -> instance will be KEPT after upload"
fi
ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << EOF
# Launch the poll+upload script DETACHED in tmux so this pipeline can finish now.
# It waits for training to complete on the VM (TRAINING_DONE/FAILED), then uploads
# the whole model dir to HF (private repo \$HF_USER/\$DATASET_NAME, created if missing),
# then deletes the Brev instance on success.
tmux kill-session -t hfupload 2>/dev/null || true
tmux new-session -d -s hfupload
tmux send-keys -t hfupload 'export PATH=\$HOME/.local/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin' C-m
tmux send-keys -t hfupload 'export UV_CACHE_DIR=/ephemeral/cache/uv' C-m
tmux send-keys -t hfupload 'bash \$HOME/poll_and_upload_hf.sh "$HF_USER" "$DATASET_NAME" "$INST" 2>&1 | tee \$HOME/poll_and_upload_hf.log' C-m
echo "poll+upload launched (tmux session hfupload). Tail: tmux capture-pane -t hfupload -p"
EOF

In [0]:
%skip
%sh
ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << EOF
# MANUAL (skipped by default): upload ONE checkpoint folder to HF, not the whole
# model. Change checkpoint-3000 to the checkpoint you want.
\$HOME/Isaac-GR00T/.venv/bin/hf upload "$HF_USER/$DATASET_NAME" \
  "/ephemeral/finetuned-models/$DATASET_NAME/checkpoint-60000" \
  checkpoint-60000 \
  --repo-type model
EOF

In [0]:
%skip
%sh
set -e

DEST="/Volumes/workspace/default/trained_models/bi_so101_clothes_250ep/"

rsync -avz --progress \
  -e "ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no" \
  "$SSH_USER@$BREV_IP:/ephemeral/finetuned-models/checkpoint-60000" \
  "$DEST/"

In [0]:
%skip
%sh
ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << 'EOF'
sudo shutdown -h now
EOF
